# Phase 7 (stretch goal): τ²-bench telecom, setup + pilot

**Why telecom:** the project proposal names "τ²-bench telecom" as the stretch goal. It is the only τ²-bench domain where the **user also runs tools** on their own phone (e.g. turn off airplane mode). The agent cannot see the phone, so it must guide the user. A runtime validator only sees the agent's tool calls, not the user's, so this is where KiL could beat the validator (Hypothesis 1).

**This notebook:** install τ²-bench (pinned version), connect Gemini, and run a **5-task pilot** to measure the real cost per dialogue. No arms yet. Cost: about $0.30.

**Folder:** `MyDrive/cmsc691-10-tau2` (separate from the airline project). Results are saved in `tau2-bench/data/simulations/`.

**Settings** (`settings_tau2.py`), kept the same as my airline arms where τ² allows it:

| Setting | τ² default | Mine | Why |
|---|---|---|---|
| Agent / user model | gpt-4.1 | gemini-3.1-flash-lite (both) | same as 05G–08 |
| Agent temperature | 0.0 | 0.0 | same |
| User temperature | 0.0 | **1.0** | same as my airline arms (τ-bench paper) |
| Max steps | 200 | **100** | τ² counts every message (agent, user, tool) as a step, and the user also acts; 100 is enough and caps cost |
| Tasks | base (114) | **test split (40)** | official τ² split; 114 × 8 is too expensive |
| Trials per task | 1 | 8 (arms), 1 (pilot) | pass^k for k = 1..8 |
| Communicate scorer | semantic LLM judge | **substring** | original τ-bench scorer; the judge needs an OpenAI key |
| Seed | 300 | 300 | same |

My numbers are not directly comparable with the τ² leaderboard, which uses the base split and gpt-4.1 as the user.

**Runtime:** CPU only (the model runs at Google). Needs the Colab Secret `GEMINI_API_KEY` (same as 05G).

In [1]:
# mount Google Drive so code and results are saved
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# get tau2-bench (pinned commit) into Drive and install it (Python 3.12, CPU)
import os
os.makedirs("/content/drive/MyDrive/cmsc691-10-tau2", exist_ok=True)
%cd /content/drive/MyDrive/cmsc691-10-tau2
if not os.path.exists("/content/drive/MyDrive/cmsc691-10-tau2/tau2-bench"):
    !git clone -q https://github.com/sierra-research/tau2-bench
%cd /content/drive/MyDrive/cmsc691-10-tau2/tau2-bench
!git fetch -q origin da0835f6c67baef58bdda18f2a3b037dfb3a1754 && git checkout -q da0835f6c67baef58bdda18f2a3b037dfb3a1754
!git log -1 --format="tau2-bench commit %h (%cd)"
!pip install -q -e .
print("installed")

/content/drive/MyDrive/cmsc691-10-tau2
/content/drive/MyDrive/cmsc691-10-tau2/tau2-bench
tau2-bench commit da0835f (Thu Oct 29 22:37:05 2026 -0700)
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for tau2 (pyproject.toml) ... done
installed


In [3]:
%%writefile /content/drive/MyDrive/cmsc691-10-tau2/settings_tau2.py
# frozen settings for every tau2-bench arm (telecom stretch goal)
# kept the same as the airline Gemini arms (settings_gemini.py) wherever tau2 allows it
MODEL = "gemini/gemini-3.1-flash-lite"   # agent and user simulator, same model as 05G-08
PRICE_IN, PRICE_OUT = 0.25, 1.50         # USD per 1M tokens, Gemini price list; for cost estimates only
TEMPERATURE = 0.0                        # agent temperature (tau2 default too)
USER_TEMPERATURE = 1.0                   # user temperature: same as my airline arms (tau2 default is 0.0)
MAX_STEPS = 100                          # tau2 default is 200; telecom needs more than tau-bench's 30
SEED = 300                               # tau2 default
CONCURRENCY = 4                          # same as airline; does not change results
DOMAIN = "telecom"
SPLIT = "test"                           # official tau2 telecom test split: 40 tasks
N_TRIALS = 8
COMMIT = "da0835f6c67baef58bdda18f2a3b037dfb3a1754"   # tau2-bench version, pinned for reproducibility

Writing /content/drive/MyDrive/cmsc691-10-tau2/settings_tau2.py


In [4]:
%%writefile /content/drive/MyDrive/cmsc691-10-tau2/analyze_tau2.py
# read a tau2 results.json: reward, turns, tokens and cost per dialogue
import json

from settings_tau2 import PRICE_IN, PRICE_OUT


def load(path):
    return json.load(open(path))


def rows(res):
    out = []
    for s in res["simulations"]:
        tok = {"agent": [0, 0], "user": [0, 0]}
        turns = {"agent": 0, "user": 0, "agent_tool": 0, "user_tool": 0}
        for m in s.get("messages") or []:
            role = m.get("role")
            who = "agent" if role == "assistant" else ("user" if role == "user" else None)
            if who:
                turns[who] += 1
                if m.get("tool_calls"):
                    turns[f"{who}_tool"] += len(m["tool_calls"])
                u = m.get("usage") or {}
                tok[who][0] += u.get("prompt_tokens") or 0
                tok[who][1] += u.get("completion_tokens") or 0
        cost = sum(i * PRICE_IN + o * PRICE_OUT for i, o in tok.values()) / 1e6
        out.append({"task": s["task_id"], "trial": s.get("trial"),
                    "reward": (s.get("reward_info") or {}).get("reward"),
                    "end": s.get("termination_reason"),
                    "agent msgs": turns["agent"], "user msgs": turns["user"],
                    "agent tool calls": turns["agent_tool"], "user tool calls": turns["user_tool"],
                    "agent in": tok["agent"][0], "agent out": tok["agent"][1],
                    "user in": tok["user"][0], "user out": tok["user"][1],
                    "cost $": round(cost, 4)})
    return out

Writing /content/drive/MyDrive/cmsc691-10-tau2/analyze_tau2.py


In [6]:
# API key from Colab Secrets (never in code), data folder, and one test call
import os, sys, json, importlib
from google.colab import userdata
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
os.environ["TAU2_DATA_DIR"] = "/content/drive/MyDrive/cmsc691-10-tau2/tau2-bench/data"
sys.path.insert(0, "/content/drive/MyDrive/cmsc691-10-tau2")
import settings_tau2 as S, analyze_tau2 as A
importlib.reload(S); importlib.reload(A)
import litellm
r = litellm.completion(model=S.MODEL, messages=[{"role": "user", "content": "Say OK."}], temperature=0)
print(S.MODEL, "->", r.choices[0].message.content.strip(), "| tokens", r.usage.prompt_tokens, "/", r.usage.completion_tokens)

gemini/gemini-3.1-flash-lite -> OK. | tokens 4 / 2


In [7]:
# how many tasks each split has (telecom = stretch goal; airline = for the free label check later)
for d in ["telecom", "airline"]:
    split = json.load(open(f"/content/drive/MyDrive/cmsc691-10-tau2/tau2-bench/data/tau2/domains/{d}/split_tasks.json"))
    print(d, {k: len(v) for k, v in split.items() if k != "full"})
test = json.load(open(f"/content/drive/MyDrive/cmsc691-10-tau2/tau2-bench/data/tau2/domains/telecom/split_tasks.json"))[S.SPLIT]
kinds = {}
for t in test:
    k = t.split("]")[0].strip("[")
    kinds[k] = kinds.get(k, 0) + 1
print("telecom", S.SPLIT, "split by issue type:", kinds)

telecom {'small': 20, 'train': 74, 'test': 40, 'base': 114}
airline {'train': 30, 'test': 20, 'base': 50}
telecom test split by issue type: {'mms_issue': 16, 'mobile_data_issue': 9, 'service_issue': 15}


In [8]:
# 5-task pilot, 1 trial: 2 MMS, 2 service, 1 mobile-data task from the test split
# (the first 5 tasks are all MMS, the hardest type, so I pick across types)
# safe to re-run: --auto-resume continues an unfinished run
import shlex
by = {}
for t in test:
    by.setdefault(t.split("]")[0].strip("["), []).append(t)
pick = by["mms_issue"][:2] + by["service_issue"][:2] + by["mobile_data_issue"][:1]
cmd = (f"tau2 run --domain {S.DOMAIN} --task-split-name {S.SPLIT} --num-trials 1 "
       f"--task-ids {' '.join(shlex.quote(t) for t in pick)} "
       f"--agent-llm {S.MODEL} --user-llm {S.MODEL} "
       f"--agent-llm-args {shlex.quote(json.dumps({'temperature': S.TEMPERATURE}))} "
       f"--user-llm-args {shlex.quote(json.dumps({'temperature': S.USER_TEMPERATURE}))} "
       f"--max-steps {S.MAX_STEPS} --seed {S.SEED} --max-concurrency {S.CONCURRENCY} "
       f"--communicate-scorer substring --save-to pilot_telecom --auto-resume")
print("\n".join(pick))
%cd /content/drive/MyDrive/cmsc691-10-tau2/tau2-bench
!{cmd}

[mms_issue]airplane_mode_on|bad_network_preference|bad_wifi_calling|break_apn_mms_setting|break_app_both_permissions|unseat_sim_card|user_abroad_roaming_enabled_off[PERSONA:Hard]
[mms_issue]airplane_mode_on|bad_network_preference|bad_wifi_calling|break_apn_mms_setting|break_app_sms_permission|data_mode_off|data_usage_exceeded|unseat_sim_card|user_abroad_roaming_disabled_on[PERSONA:None]
[service_issue]airplane_mode_on|break_apn_settings|contract_end_suspension|lock_sim_card_pin[PERSONA:None]
[service_issue]airplane_mode_on|break_apn_settings|contract_end_suspension|lock_sim_card_pin|unseat_sim_card[PERSONA:Easy]
[mobile_data_issue]airplane_mode_on|bad_network_preference|bad_vpn|data_mode_off|data_saver_mode_on|data_usage_exceeded|user_abroad_roaming_enabled_off[PERSONA:Easy]
/content/drive/MyDrive/cmsc691-10-tau2/tau2-bench
2026-10-31 18:20:11.858 | DEBUG    | tau2.registry:<module>:297 - Voice dependencies not installed, skipping voice user registration
╭────────────────────────── Sim

In [9]:
# pilot result: reward, length, tokens, cost; then the cost of one full arm
import pandas as pd
from IPython.display import display
res = A.load(f"/content/drive/MyDrive/cmsc691-10-tau2/tau2-bench/data/simulations/pilot_telecom/results.json")
df = pd.DataFrame(A.rows(res))
display(df)
n = len(df)
per = df["cost $"].mean()
print(f"dialogues: {n} | pass^1 (pilot): {df['reward'].mean():.2f} | cost per dialogue: ${per:.4f}")
print(f"ends: {df['end'].value_counts().to_dict()}")
arm = 40 * S.N_TRIALS
print(f"one arm = 40 tasks x {S.N_TRIALS} trials = {arm} dialogues -> about ${per * arm:.2f}")
for name, k in [("baseline + validator", 2), ("+ shallow", 3), ("+ semi-deep", 4)]:
    print(f"  {name:22s} {k} arms -> about ${per * arm * k:.2f}")

,task,trial,reward,end,agent msgs,user msgs,agent tool calls,user tool calls,agent in,agent out,user in,user out,cost $
0,[service_issue]airplane_mode_on|break_apn_sett...,0,0.0,user_stop,7,3,4,0,46480,385,10741,149,0.0151
1,[mobile_data_issue]airplane_mode_on|bad_networ...,0,0.0,user_stop,15,11,10,6,118775,749,47762,369,0.0433
2,[mms_issue]airplane_mode_on|bad_network_prefer...,0,0.0,user_stop,25,11,18,4,207451,1148,47007,487,0.0661
3,[service_issue]airplane_mode_on|break_apn_sett...,0,1.0,user_stop,22,18,13,9,192799,1036,79145,514,0.0703
4,[mms_issue]airplane_mode_on|bad_network_prefer...,0,0.0,user_stop,18,12,12,6,141338,807,49110,378,0.0494


dialogues: 5 | pass^1 (pilot): 0.20 | cost per dialogue: $0.0488
ends: {'user_stop': 5}
one arm = 40 tasks x 8 trials = 320 dialogues -> about $15.63
  baseline + validator   2 arms -> about $31.26
  + shallow              3 arms -> about $46.89
  + semi-deep            4 arms -> about $62.52


## Summary
- **Install:** tau2-bench commit `da0835f`, Python 3.12 (Colab CPU); test call OK: `gemini/gemini-3.1-flash-lite` → "OK."
- **Telecom test split:** 40 tasks; by issue type: MMS 16, service 15, mobile data 9.
- **Pilot (5 tasks × 1; 2 MMS, 2 service, 1 mobile data):** pass^1 = 0.20 (1 of 5, a service task); end reasons: user stop 5/5 (no errors, no step limit); agent / user messages per dialogue ≈ 17 / 11; user tool calls per dialogue ≈ 5 (agent ≈ 11). Partial credit: 12 of 26 required write actions were done (46%), so the agent fixes some issues but rarely all of them.
- **Cost:** $0.049 per dialogue (airline was about $0.016; telecom dialogues are about 3× longer and the user also calls tools) → one arm (320 dialogues) ≈ $15.6.
- **Decision:** with $30 more credit, run 3 arms × 40 tasks × 8 trials (baseline, validator, shallow), ≈ $50. Semi-deep is left out: in telecom most fixes are done by the user, and semi-deep only checks the agent's own write tools. Run baseline first and check its real cost and pass^1 before the next arm.

**Next:** 12, build the telecom policy graph from `main_policy.md` and `tech_support_manual.md` (agent-side data rules, user-side troubleshooting steps, order rules) and run the three arms; later, a free check of the τ² airline answer key against my 8 label-conflict tasks.